# H2 Phase 3 — IAM Aachen Fine-tuning

**Accelerator:** GPU T4 x1

Phase 2'nin pretrained checkpoint'inden başlayarak IAM Aachen train set'i üzerinde fine-tune eder.

**Tahmini süre:** ~4-5 saat (T4, 50 epoch / early stopping)

**Önkoşullar:**
1. IAM dataset → `Add Data → Your Datasets → paper_traning_data`
2. Phase 2 checkpoint → `Add Data → Your Datasets → pretrain_best` (Phase 2'den indirdikten sonra)

**Çıktı:** `results/phase3_results.json` — Test WA, Wilson CI, McNemar p

---
**KURALLAR (KURALLAR.md):** Test set'e sadece bu notebook'ta, bir kez bakılır. Cherry-pick yok.

---
**Donanım (makale için):**
- GPU: NVIDIA Tesla T4, 15360 MiB VRAM
- CPU: Intel(R) Xeon(R) CPU @ 2.20GHz
- RAM: 13 GB
- Env: Kaggle Notebooks, Python 3.10, PyTorch 2.3.0+cu121

In [ ]:
# Hücre 1: GPU + donanım bilgisi (makale için kaydet)
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"VRAM: {vram:.0f} GB ({int(vram*1024)} MiB)")
    print(f"PyTorch: {torch.__version__}")

!cat /proc/cpuinfo | grep 'model name' | head -1
!free -h | grep Mem
!python --version

In [ ]:
# Hücre 2: crnn-h2-code dataset'ten kopyala + bağımlılıklar
import sys, os, shutil

CODE_INPUT = "/kaggle/input/datasets/brht25/crnn-h2-code"

if os.path.exists(CODE_INPUT):
    os.makedirs("/kaggle/working/cloud", exist_ok=True)
    for fname in os.listdir(f"{CODE_INPUT}/cloud"):
        if fname.endswith((".py", ".txt", ".sh")):
            shutil.copy(f"{CODE_INPUT}/cloud/{fname}", f"/kaggle/working/cloud/{fname}")
    # aachen_splits (Phase 3 için zorunlu)
    shutil.copytree(f"{CODE_INPUT}/aachen_splits", "/kaggle/working/aachen_splits", dirs_exist_ok=True)
    shutil.copy(f"{CODE_INPUT}/trigram_lm.py", "/kaggle/working/trigram_lm.py")
    print("Scripts + aachen_splits kopyalandı OK")
else:
    print(f"⚠️  {CODE_INPUT} bulunamadı — Add Data → Your Datasets → crnn-h2-code")

sys.path.insert(0, "/kaggle/working")
os.chdir("/kaggle/working")
!pip install -q -r cloud/requirements.txt

In [ ]:
# Hücre 3: IAM dataset ve checkpoint path'lerini bul
import os, subprocess

# ── IAM Dataset: brht25/paper-traning-data ───────────────────────────────────
BASE = "/kaggle/input/paper-traning-data"

IAM_CANDIDATES_WORDS = [
    f"{BASE}/iam_words/words.txt",
    f"{BASE}/archive/iam_words/words.txt",
    f"{BASE}/words.txt",
    f"{BASE}/archive/words.txt",
]
IAM_CANDIDATES_IMGS = [
    f"{BASE}/iam_words/words",
    f"{BASE}/archive/iam_words/words",
    f"{BASE}/words",
    f"{BASE}/archive/words",
]

IAM_WORDS_TXT = next((p for p in IAM_CANDIDATES_WORDS if os.path.exists(p)), None)
IAM_WORDS_DIR = next((p for p in IAM_CANDIDATES_IMGS if os.path.exists(p)), None)

if IAM_WORDS_TXT and IAM_WORDS_DIR:
    print(f"IAM words.txt : {IAM_WORDS_TXT}")
    print(f"IAM words/    : {IAM_WORDS_DIR}")
else:
    # Klasör yapısını göster — hangi path doğru bunu anlamak için
    print("⚠️ Otomatik bulunamadı. Dataset içeriği:")
    r = subprocess.run(["find", BASE, "-maxdepth", "3", "-name", "words.txt"],
                       capture_output=True, text=True)
    print("  words.txt:", r.stdout.strip() or "(bulunamadı)")
    r2 = subprocess.run(["ls", BASE], capture_output=True, text=True)
    print("  Kök:", r2.stdout.strip())
    # Bulunan path'i buraya yaz:
    IAM_WORDS_TXT = f"{BASE}/iam_words/words.txt"   # ← gerekirse düzelt
    IAM_WORDS_DIR = f"{BASE}/iam_words/words"        # ← gerekirse düzelt
    print(f"\nManuel path kullanılıyor:\n  {IAM_WORDS_TXT}\n  {IAM_WORDS_DIR}")

# ── Pretrain checkpoint ──────────────────────────────────────────────────────
CKPT_CANDIDATES = [
    "/kaggle/input/pretrain-best/pretrain_best.pth",
    "/kaggle/input/h2-pretrain/pretrain_best.pth",
    "/kaggle/working/checkpoints/pretrain_best.pth",
]
PRETRAIN_CKPT = next((p for p in CKPT_CANDIDATES if os.path.exists(p)), None)

if PRETRAIN_CKPT:
    print(f"\nPretrain ckpt : {PRETRAIN_CKPT}")
else:
    print("\n⚠️ pretrain_best.pth bulunamadı!")
    print("Phase 2 çıktısını 'Add Data → Your Datasets' ile ekle")
    PRETRAIN_CKPT = CKPT_CANDIDATES[0]

# Working dir'e kopyala (read-only input'tan working'e)
CKPT_DIR = "/kaggle/working/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)
if os.path.exists(PRETRAIN_CKPT):
    import shutil
    shutil.copy(PRETRAIN_CKPT, f"{CKPT_DIR}/pretrain_best.pth")
    print(f"Checkpoint kopyalandı → {CKPT_DIR}/pretrain_best.pth")

In [ ]:
# Hücre 4: Fine-tuning başlat
# Bu hücre ~4-5 saat sürer. Çıktıyı takip edebilirsin.
MODEL_DIR = "/kaggle/working/Model_aachen_v3_pretrained"

!python cloud/phase3_finetune.py \
    --epochs 50 \
    --batch 128 \
    --lr 1e-4 \
    --cnn-freeze 5 \
    --patience 15 \
    --ckpt-dir {CKPT_DIR} \
    --model-dir {MODEL_DIR} \
    --iam-words {IAM_WORDS_TXT} \
    --iam-root {IAM_WORDS_DIR}

In [ ]:
# Hücre 5: Sonuçları oku ve özetle
import json, os

results_path = "/kaggle/working/results/phase3_results.json"
if os.path.exists(results_path):
    with open(results_path) as f:
        r = json.load(f)

    print("=" * 50)
    print(" PHASE 3 SONUÇLARI")
    print("=" * 50)
    print(f" Test WA          : {r['test_wa_pct']:.2f}%")
    ci = r['wilson_95ci_pct']
    print(f" Wilson 95% CI    : [{ci[0]:.2f}%, {ci[1]:.2f}%]")
    print(f" Test CER         : {r['test_cer_pct']:.2f}%")
    print(f" N samples        : {r['n_samples']:,}")

    mn = r.get('mcnemar_vs_v3_base', {})
    if mn:
        print(f"\n McNemar (V3-base vs V3-pretrained)")
        print(f" Baseline WA      : {mn.get('baseline_wa_pct', 'N/A'):.2f}%")
        print(f" Delta pp         : {mn.get('delta_pp', 'N/A'):+.2f}pp")
        print(f" McNemar p        : {mn.get('mcnemar_p', 'N/A'):.2e}")
        print(f" Significant p<.01: {'YES ✓' if mn.get('significant_p01') else 'NO'}")
    print("=" * 50)
    print("\n--- Arkadaşına gönder ---")
    print(f"Test WA  : {r['test_wa_pct']:.2f}%")
    print(f"Wilson CI: [{ci[0]:.2f}%, {ci[1]:.2f}%]")
    if mn:
        print(f"McNemar p: {mn.get('mcnemar_p', 'N/A'):.2e}")
        print(f"Delta pp : {mn.get('delta_pp', 'N/A'):+.2f}pp")
else:
    print(f"⚠️ {results_path} bulunamadı — Phase 3 tamamlandı mı?")

In [ ]:
# Hücre 6: Checkpoint boyutu ve indirme yolu
import os
best_wa = f"{MODEL_DIR}/best_model_wa.pth"
if os.path.exists(best_wa):
    mb = os.path.getsize(best_wa) / 1024**2
    print(f"best_model_wa.pth: {mb:.1f} MB")
    print(f"Output sekmesinden indir: {MODEL_DIR}/")
else:
    print("Checkpoint bulunamadı")